In [38]:
# 匯入模組
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.manifold import TSNE

In [39]:
data = pd.read_csv('BackpackPrediction_train.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    300000 non-null  int64  
 1   Brand                 290295 non-null  object 
 2   Material              291653 non-null  object 
 3   Size                  293405 non-null  object 
 4   Compartments          300000 non-null  float64
 5   Laptop Compartment    292556 non-null  object 
 6   Waterproof            292950 non-null  object 
 7   Style                 292030 non-null  object 
 8   Color                 290050 non-null  object 
 9   Weight Capacity (kg)  299862 non-null  float64
 10  Price                 300000 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 25.2+ MB


In [40]:
# 刪除類別變項中不含資料的 id
categorical_features = ['Brand', 'Material', 'Size', 'Laptop Compartment', 'Waterproof', 'Style', 'Color']
missing_categorical = data[categorical_features].isnull().any(axis=1)
data = data[~missing_categorical]
data.info() # 再檢查一次是否有缺失值
data.head()

<class 'pandas.core.frame.DataFrame'>
Index: 246686 entries, 0 to 299999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    246686 non-null  int64  
 1   Brand                 246686 non-null  object 
 2   Material              246686 non-null  object 
 3   Size                  246686 non-null  object 
 4   Compartments          246686 non-null  float64
 5   Laptop Compartment    246686 non-null  object 
 6   Waterproof            246686 non-null  object 
 7   Style                 246686 non-null  object 
 8   Color                 246686 non-null  object 
 9   Weight Capacity (kg)  246686 non-null  float64
 10  Price                 246686 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 22.6+ MB


,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price
0,0,Jansport,Leather,Medium,7.0,Yes,No,Tote,Black,11.611723,112.15875
1,1,Jansport,Canvas,Small,10.0,Yes,Yes,Messenger,Green,27.078537,68.88056
2,2,Under Armour,Leather,Small,2.0,Yes,No,Messenger,Red,16.643760,39.17320
3,3,Nike,Nylon,Small,8.0,Yes,No,Messenger,Green,12.937220,80.60793
4,4,Adidas,Canvas,Medium,1.0,Yes,Yes,Messenger,Green,17.749338,86.02312


In [41]:
data = data.drop(columns=['id'])

In [42]:
# one-hot coding 以及將布林值轉為0/1
data_encoded = pd.get_dummies(data, drop_first=True)
bool_cols = data_encoded.select_dtypes(include=['bool']).columns
data_encoded[bool_cols] = data_encoded[bool_cols].astype(int)

In [43]:
X = data_encoded
# 提取Price
y = data_encoded['Price']

In [44]:
# 訓練集、測試集切分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [45]:
# 標準化數據
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [50]:
# Apply PCA
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.fit_transform(X_test_scaled)

In [ ]:
# 建立隨機森林模型
model = RandomForestRegressor(n_estimators=100  , max_depth=10, min_samples_leaf=10, min_samples_split=10)
model.fit(X_train_pca, y_train)

# 預測
y_pred = model.predict(X_test_pca)

# 模型評估
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
accuracy = model.score(X_test_pca, y_test)

print("模型評估結果：")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R²: {r2}")
print(f'accuracy: {accuracy}')

模型評估結果：
MSE: 5377.604442075201
RMSE: 73.33215148947426
R²: -2.5576683972695653
accuracy: -2.5576683972695653


In [52]:
# 讀取測試數據
test_data = pd.read_csv('BackpackPrediction_test.csv')

# 保留 Id 供最終輸出
test_ids = test_data['id'] 
# 移除 Id 欄位（若存在）
test_data.drop(columns=['id'], inplace=True)

# 處理缺失值（填充數值型變數的中位數，類別變數填充 'None'）
for col in test_data.columns:
    if test_data[col].dtype == "object":
        test_data[col].fillna("None", inplace=True)
    else:
        test_data[col].fillna(0, inplace=True)

# one-hot coding 以及將布林值轉為0/1
test_data_encoded = pd.get_dummies(test_data, drop_first=True)
bool_cols = test_data_encoded.select_dtypes(include=['bool']).columns
test_data_encoded[bool_cols] = test_data_encoded[bool_cols].astype(int)

# 確保測試數據的欄位與訓練數據匹配（若有缺少的欄位則補0）
missing_cols = set(X.columns) - set(test_data_encoded.columns)
for col in missing_cols:
    test_data_encoded[col] = 0

# 確保欄位順序一致
test_data_encoded = test_data_encoded[X.columns]


In [53]:
X_test_data = test_data_encoded.copy()

# 2. 進行標準化（使用訓練階段 fit 的 scaler）
X_test_scaled = scaler.transform(X_test_data)

# 3. 利用訓練時 fit 好的 PCA 模型進行降維
X_test_pca = pca.transform(X_test_scaled)

# 4. 接下來就可以使用 rf_model 進行預測，X_test_pca 的 shape 應為 (樣本數, 17)
test_predictions = model.predict(X_test_pca)


In [54]:
# 建立提交結果
submission = pd.DataFrame({"id": test_ids, "Price": test_predictions})
submission.set_index("id", inplace=True)

print(submission)

submission.to_csv('BackpackPricePred_PCA+RF_V2.csv')

             Price
id                
300000  129.766242
300001  135.442750
300002  137.585260
300003  148.616477
300004  148.520767
...            ...
499995  147.611055
499996  136.937139
499997  133.805154
499998  147.962599
499999  145.334174

[200000 rows x 1 columns]
